In [1]:
import torch
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [2]:
import sys
sys.path.append('../src')
from model import GeosteeringHybridModel

In [3]:
# Load the model
WINDOW_SIZE = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GeosteeringHybridModel(num_features=2, window_size=WINDOW_SIZE)

model.load_state_dict(torch.load("../src/models/baseline_geosteering_model.pth", map_location=device))
model.to(device)
model.eval()

# 3. Load test data
test_well_id = "000d7d20"
df_test = pd.read_csv(f"../data/test/{test_well_id}__horizontal_well.csv")

# fill holes in Gamma Ray (Forward Fill & Backward Fill)
df_test['GR'] = df_test['GR'].ffill().bfill()
df_test['Z'] = df_test['Z'].ffill().bfill()

# 4. Features extraction
features = df_test[['GR', 'Z']].values
features[:, 0] = features[:, 0] / 150.0   # GR scaling
features[:, 1] = features[:, 1] / 10000.0 # Z scaling

# Prepare array for predictions
predictions = np.full(len(df_test), np.nan)

# 5. Sliding Window over the whole dataset
with torch.no_grad():
    for i in range(len(features) - WINDOW_SIZE):
        # Cut out window
        window = features[i : i + WINDOW_SIZE]

        # Transform into pytorch tensor and adjust dimensions (Batch=1, Channels=2, Seq=50)
        x_tensor = torch.tensor(window, dtype=torch.float32).transpose(0, 1).unsqueeze(0).to(device)

        # predictions
        pred_tvt = model(x_tensor)

        # Save prediction in the right position
        predictions[i + WINDOW_SIZE - 1] = pred_tvt.item()

In [ ]:
# 6. Write predictions into the dataframe
df_test['TVT_pred'] = predictions
print("Vorhersagen erfolgreich in den DataFrame geschrieben!")
# 7. Visualize the results
plt.figure(figsize=(12, 6))
print("Figure ")
# Known TVT (Ground Truth for the first part)
plt.plot(df_test['MD'], df_test['TVT_input'], color='green', linewidth=2, label='Bekanntes TVT_input')
print("Figure Kopfzeile")
# Our model prediction
plt.plot(df_test['MD'], df_test['TVT_pred'], color='purple', linestyle='--', linewidth=2, label='Modell Vorhersage (TVT)')
print("Figure Modell Predictions")

# Start of evaluation period
print("Start Evaluation")
eval_start_idx = df_test['TVT_input'].isna().idxmax()
eval_start_md = df_test.loc[eval_start_idx, 'MD']
print("End Evaluation")
print("Plot Start")
plt.axvline(x=eval_start_md, color='red', linestyle='-', alpha=0.5, label='Start Blindflug (NaN)')
plt.axvspan(eval_start_md, df_test['MD'].max(), color='red', alpha=0.1)
plt.title(f'Geosteering Vorhersage für Bohrloch {test_well_id}')
plt.xlabel('Measured Depth (MD)')
plt.ylabel('True Vertical Thickness (TVT)')
plt.legend()
plt.grid(True, alpha=0.3)
print("Starte tight_layout...")
plt.tight_layout()
print("Saving started ....")
plt.savefig('inference_plot_000d7d20.png', dpi=150, bbox_inches='tight')
plt.close('all')
print("Fertig! Skript beendet.")


Vorhersagen erfolgreich in den DataFrame geschrieben!
Figure 
Figure Kopfzeile
Figure Modell Predictions
Start Evaluation
End Evaluation
Plot Start
